# DistilBERT Classifier

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append('..')
from src.data_utils import stratified_split, build_label_maps, encode_labels
from src.train_utils import set_seed, compute_metrics
from src.pytorch_models import select_device

set_seed(2026)
device = select_device()
print('Device:', device)

CLASS_ORDER = ['Human', 'Google', 'Meta', 'OpenAI', 'Anthropic']

CHECKPOINT     = 'distilbert-base-uncased'
TRAIN_FRACTION = 0.1   # Bert é mais lento
MAX_LEN        = 128
BATCH_SIZE     = 16
LR             = 2e-5
EPOCHS         = 3
PATIENCE       = 2

## Dados

In [ ]:
df = pd.read_csv('../data/dataset_final.csv')
train_df, val_df = stratified_split(df, 'text', 'label', test_size=0.2, seed=2026)

if TRAIN_FRACTION < 1.0:
    train_df = train_df.groupby('label', group_keys=False).apply(
        lambda g: g.sample(frac=TRAIN_FRACTION, random_state=2026)
    ).reset_index(drop=True)

test_df = pd.read_csv('../data/subm1_labels_revealed.csv', sep=';')
test_df = test_df.rename(columns={'Text': 'text', 'Label': 'label'})

print(f'Treino: {len(train_df):,} | Validação: {len(val_df):,} | Teste (prof): {len(test_df)}')

label_to_idx, idx_to_label = build_label_maps(CLASS_ORDER)
y_train = encode_labels(train_df['label'], label_to_idx)
y_val   = encode_labels(val_df['label'],   label_to_idx)
y_test  = encode_labels(test_df['label'],  label_to_idx)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

class TextDataset(Dataset):
    
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = TextDataset(train_df['text'], y_train, tokenizer, MAX_LEN)
val_ds   = TextDataset(val_df['text'],   y_val,   tokenizer, MAX_LEN)
test_ds  = TextDataset(test_df['text'],  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Modelo

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(CLASS_ORDER),
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros totais: {total_params:,} | Treináveis: {trainable:,}')

## Treino

In [ ]:
# Loop de treino próprio — run_training não suporta o formato (input_ids, attention_mask, labels)

def train_one_epoch_bert(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()

        total_loss += outputs.loss.item() * labels.size(0)
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / total, correct / total


def evaluate_bert(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            preds   = outputs.logits.argmax(dim=1)

            total_loss += outputs.loss.item() * labels.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_true.extend(labels.cpu().tolist())
            all_pred.extend(preds.cpu().tolist())
    return total_loss / total, correct / total, all_true, all_pred

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_state, best_val, best_epoch = None, -1.0, -1
no_improve = 0

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch_bert(model, train_loader, optimizer, device)
    va_loss, va_acc, _, _ = evaluate_bert(model, val_loader, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f} val_acc={va_acc:.4f}')

    if va_acc > best_val:
        best_val   = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stopping (best epoch={best_epoch}, best val_acc={best_val:.4f})')
        break

model.load_state_dict(best_state)
print(f'\nMelhor epoch: {best_epoch}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (tr, vl, title) in zip(axes, [
    ('train_loss', 'val_loss', 'Loss'),
    ('train_acc',  'val_acc',  'Accuracy'),
]):
    ax.plot(history[tr], label='treino')
    ax.plot(history[vl], label='validação')
    ax.axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'best epoch ({best_epoch})')
    ax.set_xlabel('Época')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Avaliação

In [ ]:
for loader, name in [(val_loader, 'Validação (20%)'), (test_loader, 'Teste (prof)')]:
    _, _, y_true, y_pred = evaluate_bert(model, loader, device)
    m = compute_metrics(y_true, y_pred, list(range(len(CLASS_ORDER))))
    print(f'\n=== {name} ===')
    print(f'Accuracy: {m["accuracy"]:.4f} | Macro F1: {m["macro_f1"]:.4f}')
    print(m['report'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (loader, name) in zip(axes, [(val_loader, 'Validação (20%)'), (test_loader, 'Teste (prof)')]):
    _, _, y_true, y_pred = evaluate_bert(model, loader, device)
    m = compute_metrics(y_true, y_pred, list(range(len(CLASS_ORDER))))
    sns.heatmap(m['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER, ax=ax)
    ax.set_title(f'DistilBERT — {name}')
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

## Guardar

In [ ]:
import os
os.makedirs('../modelos/distilbert', exist_ok=True)

model.save_pretrained('../modelos/distilbert')
tokenizer.save_pretrained('../modelos/distilbert')
print('Guardado em ../modelos/distilbert/')